# 01 – Visión general y limpieza de datos

Vamos a explorar y limpiar el conjunto de datos de videojuegos de Steam
que utilizaremos como base para nuestro sistema de recomendación.

Se quiere dejar claro:

- Qué contiene el dataset (qué representa cada columna).
- Qué decisiones tomamos sobre limpieza y transformación de datos.
- Qué columnas son relevantes para el **MVP** (modelo mínimo viable) de recomendador,
  y cuáles descartamos porque no aportan valor directo.

Este archivo será el punto de partida para el resto del proyecto:

- Análisis exploratorio y visualizaciones.
- Modelos de recomendación basados en contenido (texto y metadatos).
- Aplicación web de recomendaciones para usuarios.

In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import json
import numpy as np
from numpy._core.defchararray import upper
import seaborn as sns
import ast
from pathlib import Path

# Opciones de visualización para ver mejor las tablas

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Estilo básico de gráficos
sns.set(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle
from sklearn.feature_selection import f_classif, SelectKBest

In [2]:
# 0. Configuración de la ruta del proyecto y del módulo src

import os
import sys
from pathlib import Path

# Este notebook está en 'notebooks/', la raíz del proyecto es un nivel por encima
PROJECT_ROOT = Path("..").resolve()

# 1) Cambiamos el directorio de trabajo a la raíz del proyecto
os.chdir(PROJECT_ROOT)
print("Directorio de trabajo actual:", Path.cwd())

# 2) Aseguramos que la raíz está en sys.path para poder importar src.*
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT añadido a sys.path:", PROJECT_ROOT)

Directorio de trabajo actual: /workspaces/Final-Project-Videogame-Recommender
PROJECT_ROOT añadido a sys.path: /workspaces/Final-Project-Videogame-Recommender


## 1. Dataset de partida

El dataset original procede de la API de Steam y de SteamSpy, y ha sido publicado y
mantenido por Fronkon Games.

Nosotros lo utilizamos a través de un archivo en formato **Parquet** alojado en Hugging Face:

`games.parquet` (≈122.611 juegos y 43 columnas)

Cada fila del dataset representa **un videojuego publicado en Steam** e incluye:

- Información básica (nombre, fecha de lanzamiento, precio…).
- Información técnica (plataformas, idiomas).
- Información de calidad/popularidad (reseñas positivas/negativas, puntuaciones, etc.).
- Información categórica (géneros, etiquetas, categorías).
- Enlaces e imágenes (web, header_image, capturas).

In [ ]:
# 1. Carga del dataset original desde Hugging Face y guardado en data/raw

import pandas as pd
from pathlib import Path

DATA_URL = "https://huggingface.co/datasets/pabloramcos/Videogame-Recommender-Final-Project/resolve/main/games.parquet"

# Intentamos usar PROJECT_ROOT si ya existe; si no, lo inferimos como la carpeta del repo
try:
    PROJECT_ROOT
except NameError:
    # Si estás en notebooks/, PROJECT_ROOT es la carpeta de arriba
    from pathlib import Path as _Path
    PROJECT_ROOT = _Path("..").resolve()

print("PROJECT_ROOT usado para datos:", PROJECT_ROOT)

# Ahora construimos una ruta ABSOLUTA: <root>/data/raw
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Leemos el parquet directamente desde la URL
df = pd.read_parquet(DATA_URL)

print("Dimensiones del dataframe original:", df.shape)
display(df.head())

# Guardamos una copia "congelada" del dataset original en <root>/data/raw/games.parquet
raw_path = raw_dir / "games.parquet"
df.to_parquet(raw_path, index=False)
print(f"Dataset bruto guardado en: {raw_path}")

PROJECT_ROOT usado para datos: /workspaces/Final-Project-Videogame-Recommender


In [ ]:
df.loc[df["name"].str.contains("Hades")]
#df.loc[df["app_id"].str.contains("1145360")]
df["app_id"] = df["app_id"].astype(int)  # o pd.to_numeric(df["app_id"])
df.loc[(df["app_id"] >= 100000) & (df["app_id"] <= 200000)]
df.loc[df["app_id"] == df["app_id"].min()]

## Exploración de datos

Comprobamos las dimensiones del dataframe y, además de si los datos concuerdan con el número total, vemos el tipo de dato y podemos separar entre variables categóricas y numéricas

In [ ]:
print(f"Dimensiones del dataframe: {df.shape}")
print(df.info())

Procedemos a contabilizar los nulos: 

In [ ]:
print(f"Valores null por columna: \n{df.isna().sum()}")

## 2. Columnas y toma de decisiones

En la siguiente tabla se resumen:

- **Columna**: nombre en el dataframe
- **Tipo de dato**: tipo de dato que viene en el dataset original
- **Descripción**: qué significa esa columna
- **¿Se conserva?**: si la mantendremos en el dataset limpio
- **Motivo principal**: por qué la mantenemos o la descartamos

> Nota: el tipo es orientativo (puede cambiar ligeramente al leer el Parquet con pandas).

| Columna                     | Tipo de dato   | Descripción breve                                                                                         | ¿Se conserva? | Motivo principal                                                                                                           |
|----------------------------|-----------------|-----------------------------------------------------------------------------------------------------------|--------------|----------------------------------------------------------------------------------------------------------------------------|
| `appID`                    | string          | Identificador único del juego en Steam.                                                                   | ✅ Sí         | Es la “clave” del juego, necesaria para cualquier referencia, joins, etc.                                                 |
| `name`                     | string          | Nombre del juego (título comercial).                                                                     | ✅ Sí         | Fundamental para mostrar recomendaciones y para búsquedas por nombre.                                                     |
| `release_date`             | string          | Fecha de lanzamiento (texto tipo “Aug 1, 2023”).                                                          | ✅ Sí         | Útil para filtrar por juegos recientes/antiguos. La convertiremos a formato fecha (`datetime`).                           |
| `estimated_owners`         | string          | Rango estimado de propietarios, ej. “0 - 20000”.                                                         | ➖ Opcional*  | Es una medida aproximada de popularidad. Puede servir para ranking, pero no es imprescindible. La podemos conservar.      |
| `peak_ccu`                 | int64           | Máximo de usuarios concurrentes (ayer).                                                                   | ✅ Sí         | Métrica de actividad reciente; puede complementar la popularidad del juego.                                               |
| `required_age`             | int64           | Edad mínima recomendada para jugar.                                                                      | ✅ Sí         | Útil para filtros (ej. evitar juegos +18).                                                                                |
| `price`                    | float64         | Precio actual en Steam (en dólares).                                                                     | ✅ Sí         | Importante para filtrar recomendaciones por presupuesto.                                                                  |
| `dlc_count`                | int64           | Número de DLCs disponibles para el juego.                                                                 | ➖ Opcional   | Puede asociarse con juegos “grandes”/complejos, pero no es crítico para el MVP. Podemos mantenerlo por si aporta señal.   |
| `detailed_description`     | string          | Descripción larga del juego (texto libre, “about the game”).                                             | ✅ Sí         | Campo **clave** para el análisis de texto (NLP) y la comprensión semántica del juego.                                     |
| `short_description`        | string          | Descripción corta mostrada en Steam.                                                                     | ✅ Sí         | Muy útil como resumen del juego, y alternativa cuando la descripción larga sea muy grande o falte.                        |
| `supported_languages`      | list            | Lista de idiomas soportados por el juego (subtítulos/ interfaz).                                         | ✅ Sí         | Permite filtrar por idioma; relevante para muchos usuarios.                                                               |
| `full_audio_languages`     | list            | Idiomas con doblaje completo de audio.                                                                   | ➖ Opcional   | Menos crítico que `supported_languages`. Podemos mantenerlo pero no es obligatorio usarlo en el MVP.                      |
| `reviews`                  | string          | Texto resumen de reseñas/valoraciones de usuarios.                                                       | ✅ Sí         | Potencialmente útil para análisis de sentimiento y enriquecimiento del contexto textual.                                  |
| `header_image`             | string (URL)    | URL de la imagen principal del juego en Steam.                                                           | ✅ Sí         | Muy valioso para la app (mostrar carátula del juego en las recomendaciones).                                              |
| `website`                  | string (URL)    | Web oficial del juego.                                                                                   | ➖ Opcional   | No es necesaria para el modelo, pero puede ser útil como enlace extra en la app.                                          |
| `support_url`              | string (URL)    | URL de soporte técnico del juego.                                                                        | ❌ No         | No aporta valor al sistema de recomendación; es información de servicio.                                                  |
| `support_email`            | string          | Email de soporte del juego/estudio.                                                                      | ❌ No         | Ídem: útil para soporte, no para recomendar juegos.                                                                       |
| `windows`                  | bool            | Indica si el juego está disponible en Windows.                                                           | ✅ Sí         | Importante para filtrar según el sistema operativo del usuario.                                                           |
| `mac`                      | bool            | Indica si el juego está disponible en macOS.                                                             | ✅ Sí         | Importante para filtrar según el sistema operativo del usuario.                                                           |
| `linux`                    | bool            | Indica si el juego está disponible en Linux.                                                             | ✅ Sí         | Importante para filtrar según el sistema operativo del usuario.                                                           |
| `metacritic_score`         | int64           | Puntuación media en Metacritic (crítica especializada).                                                  | ✅ Sí         | Buena señal de calidad del juego, complementa las valoraciones de usuarios.                                               |
| `metacritic_url`           | string (URL)    | Enlace a la página de Metacritic.                                                                        | ❌ No         | El enlace en sí no aporta nada al modelo; se puede recuperar a partir del score si hiciera falta.                         |
| `user_score`               | int64           | Puntuación agregada por usuarios según SteamSpy (0–100 aprox.).                                          | ✅ Sí         | Señal directa de satisfacción de usuarios; muy útil para re-ordenar recomendaciones.                                      |
| `positive`                 | int64           | Número de reseñas positivas.                                                                             | ✅ Sí         | Parte de la señal de sentimiento/calidad popular; combinable con `negative`.                                             |
| `negative`                 | int64           | Número de reseñas negativas.                                                                             | ✅ Sí         | Permite calcular ratios de satisfacción (ej. % positivo).                                                                 |
| `score_rank`               | string          | Ranking textual del juego (ej. “Top 10%”).                                                               | ➖ Opcional   | Redundante con otras métricas (user_score, positive/negative). Podemos eliminarla o mantenerla solo como referencia.     |
| `achievements`             | int64           | Número de logros (achievements) disponibles.                                                             | ➖ Opcional   | Puede correlacionarse con profundidad del juego, pero no es crítico para el MVP.                                          |
| `recommendations`          | int64           | Número de recomendaciones en Steam (usuarios que “recomiendan” el juego).                               | ✅ Sí         | Excelente proxy de popularidad/aceptación de los jugadores.                                                               |
| `notes`                    | string          | Anotaciones varias (información no estructurada, a veces vacía).                                        | ❌ No         | Campo ruidoso y poco estandarizado; no aporta valor claro al modelo.                                                      |
| `average_playtime_forever` | int64           | Tiempo medio de juego total (en minutos) entre los propietarios.                                        | ✅ Sí         | Indica “engagement” a largo plazo; útil como señal extra para priorizar títulos.                                          |
| `average_playtime_2weeks`  | int64           | Tiempo medio de juego en las últimas 2 semanas.                                                          | ➖ Opcional   | Menos relevante para un análisis histórico; podemos descartarlo para simplificar.                                         |
| `median_playtime_forever`  | int64           | Tiempo mediano de juego total (en minutos).                                                             | ➖ Opcional   | Similar a `average_playtime_forever`; podríamos quedarnos solo con uno de los dos.                                       |
| `median_playtime_2weeks`   | int64           | Tiempo mediano de juego en las últimas 2 semanas.                                                       | ❌ No         | Campo muy específico, aporta poco para el MVP.                                                                            |
| `developers`               | list            | Lista de estudios/desarrolladores del juego.                                                             | ✅ Sí         | Puede ser útil para filtrar por estudio o detectar gustos por determinados desarrolladores.                               |
| `publishers`               | list            | Lista de publishers del juego.                                                                           | ➖ Opcional   | Similar a `developers`; se puede mantener pero no es crítico para el MVP.                                                |
| `categories`               | list            | Categorías de Steam (ej. “Single-player”, “Co-op”, “Steam Achievements”).                                | ✅ Sí         | Ayuda a entender el tipo de experiencia (singleplayer, cooperativo, etc.) y filtrar por ello.                             |
| `genres`                   | list            | Géneros principales del juego (ej. “RPG”, “Action”, “Indie”).                                            | ✅ Sí         | Uno de los campos **más importantes** para recomendaciones basadas en contenido.                                          |
| `tags`                     | list / string   | Etiquetas detalladas del juego (ej. “roguelike”, “pixel graphics”, “difficult”).                        | ✅ Sí         | Campo clave para conectar gustos del usuario (texto libre) con juegos mediante NLP/embeddings.                           |
| `screenshots`              | list (URLs)     | URLs a múltiples capturas de pantalla del juego.                                                        | ➖ Opcional   | Útil visualmente, pero consume bastante espacio. Para el MVP basta con `header_image`.                                   |
| `movies`                   | list            | Información sobre vídeos/trailers del juego.                                                             | ❌ No         | Poco práctico de usar en este proyecto, y aumenta tamaño/ruido del dataset.                                               |
| `packages`                 | string          | Información de paquetes comerciales (bundles) donde aparece el juego.                                   | ❌ No         | Poco relevante para las recomendaciones de gustos; información comercial interna de Steam.                                |
| `__index_level_0__` etc.   | int64 / misc    | Columnas técnicas de índice que pueden aparecer al guardar/cargar Parquet (no presentes en el dataset original). | ❌ No  | Son artefactos de guardado, no representan información del juego y se eliminarán en la limpieza.                         |

\* “Opcional” significa que, aunque no sean imprescindibles para el MVP, podemos conservarlas en
el dataset limpio si el tamaño y la complejidad se mantienen manejables. En caso de duda, priorizaremos:

- **Texto + géneros + tags + categorías** para el modelo de NLP.
- **Precio + plataformas + idiomas** para filtros del usuario.
- **User_score + positive/negative + recommendations + playtime** como señales de calidad/popularidad.

## Columnas clave para recomendaciones:

- name
- price
- release_date
- windows/mac/linux
- short_description
- detailed_description
- reviews
- genres
- categories
- tags
- user_score
- positive
- negative
- recommendations

In [ ]:
# 2. Resumen automático de columnas: tipo, % de nulos, nº de únicos, ejemplo de valor

# Empezamos con tipo y % de nulos
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_frac": df.isna().mean()
})

# Calculamos n_unique columna a columna para evitar errores con arrays/listas
n_unique = {}
for col in df.columns:
    try:
        # Intento directo
        n_unique[col] = df[col].nunique()
    except TypeError:
        # Si la columna tiene objetos no "hasheables" (arrays, listas, etc.),
        # la convertimos todo a string y contamos únicos sobre el texto
        n_unique[col] = df[col].astype(str).nunique()

summary["n_unique"] = pd.Series(n_unique)

# Añadimos un ejemplo de valor por columna (primera fila no nula)
example_values = {}
for col in df.columns:
    non_null_series = df[col].dropna()
    example_values[col] = non_null_series.iloc[0] if len(non_null_series) > 0 else np.nan

summary["example_value"] = pd.Series(example_values)

# Ordenamos por % de nulos descendente sólo para inspección
summary_sorted = summary.sort_values("missing_frac", ascending=False)

print("Resumen de columnas (tipo, % nulos, nº valores únicos, ejemplo):")
display(summary_sorted)

print("\nLista de columnas del dataframe:")
print(list(df.columns))

## 3. Criterios generales de limpieza

Antes de entrar en el detalle de cada columna, definimos unas reglas generales:

1. **Mantener filas que representen juegos reales y jugables**  
   - Queremos centrarnos en juegos completos (no DLCs, bandas sonoras, etc.).  
   - En el dataset original ya se han filtrado la mayoría de estos casos, pero si detectamos
     entradas claramente “vacías” o con datos mínimos, podremos eliminarlas.

2. **Conservar información útil para el recomendador**  
   Prioridad alta para:
   - Descripciones de texto (`detailed_description`, `short_description`, `reviews`).
   - Métricas de calidad (`user_score`, `positive`, `negative`, `recommendations`, `metacritic_score`).
   - Metadatos de contenido (`genres`, `tags`, `categories`).
   - Filtros frecuentes (`price`, `release_date`, `windows`, `mac`, `linux`, `supported_languages`).

3. **Eliminar columnas de soporte o muy específicas**  
   - Campos pensados para soporte al cliente (`support_url`, `support_email`).
   - Enlaces secundarios (`metacritic_url`).
   - Información comercial interna (`packages`).
   - Campos técnicos generados al exportar a Parquet (índices).

4. **Estandarizar tipos**  
   - Convertir fechas a tipo `datetime`.
   - Asegurarnos de que precios y puntuaciones son numéricos.
   - Convertir listas codificadas como texto (como `tags` cuando vienen en formato JSON/dict) a listas Python limpias.

5. **Tratar valores nulos de forma coherente**  
   - En campos críticos para el modelo (texto descriptivo), intentaremos rellenar usando otras columnas
     (por ejemplo, usar `short_description` si falta `detailed_description`).
   - Si un juego no tiene ninguna descripción textual utilizable, probablemente se eliminará del dataset
     que se usará para el modelo de recomendación basado en NLP.

In [ ]:
# 3. Creación de df_clean y primeras operaciones de limpieza básica

# Trabajamos sobre una copia para no modificar el df original
df_clean = df.copy()

print("Dimensiones originales del dataframe:", df.shape)

# --- 3.1 Comprobamos duplicados por appID (identificador del juego) ---

if "appID" in df_clean.columns:
    # Duplicados considerando solo la columna appID
    dup_appid_mask = df_clean.duplicated(subset="appID")
    n_dup_appid = dup_appid_mask.sum()
    print(f"\nNúmero de filas con appID duplicado: {n_dup_appid}")

    if n_dup_appid > 0:
        print("Ejemplo de juegos con appID duplicado:")
        display(df_clean.loc[dup_appid_mask, ["appID", "name"]].head())

        # Eliminamos filas duplicadas por appID, conservando la primera aparición
        df_clean = df_clean.drop_duplicates(subset="appID", keep="first").copy()
        print("Dimensiones después de eliminar duplicados por appID:", df_clean.shape)
else:
    print("\nAviso: no se encontró la columna 'appID' para controlar duplicados por identificador.")


# --- 3.2 Normalización básica de algunas columnas de texto ---

# Columnas de texto importantes que usaremos luego
text_cols = [
    "name",
    "detailed_description",
    "short_description",
    "reviews"
]

for col in text_cols:
    if col in df_clean.columns:
        # Convertimos a string y quitamos espacios en blanco al principio y al final
        df_clean[col] = (
            df_clean[col]
            .astype("string")  # tipo string de pandas (permite nulos)
            .str.strip()
        )

print("\nTipos de datos tras las primeras normalizaciones (primeras 20 columnas):")
display(df_clean.dtypes.to_frame("dtype").head(20))

print("\nVista rápida de algunas columnas de texto normalizadas:")
cols_preview = [c for c in text_cols if c in df_clean.columns]
display(df_clean[cols_preview].head())

## 4. Cómo tratamos los valores nulos en las descripciones

Nuestro sistema de recomendación va a usar **NLP (procesamiento del lenguaje natural)** para
entender los gustos del usuario y el contenido de cada juego. Por eso, las columnas de texto
descriptivo son críticas:

- `detailed_description` – descripción larga.
- `short_description` – descripción corta.
- `reviews` – texto resumen de reseñas de usuarios.

### 4.1 Objetivo

Para cada juego, queremos construir un **texto representativo** que describa el juego lo mejor posible.
Lo ideal es contar al menos con una de estas columnas (o una combinación de ellas).

### 4.2 Estrategia de limpieza de nulos

1. **Si existe `detailed_description` (no nula y con algo de longitud)**  
   > Usamos `detailed_description` como texto principal.

2. **Si `detailed_description` está vacía o es muy corta**, pero sí hay `short_description`:  
   > Usamos `short_description` como texto principal.

3. **Si ambas (`detailed_description` y `short_description`) están vacías**, pero hay texto en `reviews`:  
   > Usaremos `reviews` como respaldo (aunque puede ser más ruidoso).

4. **Si las tres (`detailed_description`, `short_description`, `reviews`) están vacías o casi vacías:**  
   - El juego **no aporta información textual para el modelo NLP**.
   - En ese caso, podemos:
     - Mantenerlo solo para un recomendador basado en géneros/tags, o
     - Eliminarlo del subconjunto de datos preparado para el modelo de texto.
   - En este notebook marcaremos estas filas para poder filtrarlas más adelante.

### 4.3 Resultado esperado

Al final de la limpieza tendremos:

- Una nueva columna (por ejemplo, `text_description`) que combine la mejor fuente de texto
  disponible para cada juego.
- Un registro de cuántos juegos no tienen texto suficiente, para decidir si:
  - Los descartamos completamente, o
  - Los usamos solo en un recomendador basada en metadatos (tags/genres) sin NLP.

In [ ]:
# 4. Tratamiento de descripciones y creación de 'text_description' (versión ligera)

TEXT_COLS = ["detailed_description", "short_description", "reviews"]

print("Columnas de texto presentes en df_clean:")
for col in TEXT_COLS:
    print(f" - {col}: {'✅' if col in df_clean.columns else '❌ NO ENCONTRADA'}")

print("\nPorcentaje de valores nulos en columnas de texto:")
for col in TEXT_COLS:
    if col in df_clean.columns:
        missing_frac = df_clean[col].isna().mean()
        print(f" - {col}: {missing_frac:.2%}")

# 4.1 Aseguramos tipo string "normal" (object) y limpiamos espacios
#     IMPORTANTE: NO usamos dtype "string" de pandas para evitar arrays Unicode gigantes.

for col in ["detailed_description", "short_description"]:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .fillna("")       # nulos -> cadena vacía
            .astype(str)      # strings tipo object, no StringArray
            .str.strip()
        )

# 'reviews' puede ser extremadamente largo; de momento NO lo concatenamos para no reventar memoria.
# Si lo necesitamos en el futuro, podremos usarlo aparte para análisis de sentimiento.

# Creamos series auxiliares (si falta alguna columna, la creamos vacía)
if "detailed_description" in df_clean.columns:
    dd = df_clean["detailed_description"]
else:
    dd = pd.Series("", index=df_clean.index, dtype=object)

if "short_description" in df_clean.columns:
    sd = df_clean["short_description"]
else:
    sd = pd.Series("", index=df_clean.index, dtype=object)

# 4.2 Construimos 'text_description' con prioridad:
#     1) detailed_description si no está vacía
#     2) si detailed está vacía, usamos short_description

# Empezamos con detailed_description
text_desc = dd.copy()

# Máscara: detailed vacío y short no vacío
mask_use_short = (text_desc.str.len() == 0) & (sd.str.len() > 0)

# Donde detailed está vacío pero short no, usamos short
text_desc[mask_use_short] = sd[mask_use_short]

# Limpiamos blancos y pasamos cadenas vacías a NaN
text_desc = text_desc.str.strip()
text_desc = text_desc.replace("", pd.NA)

df_clean["text_description"] = text_desc

# Creamos la columna booleana 'has_text'
df_clean["has_text"] = df_clean["text_description"].notna()

# Proporción de juegos con texto utilizable
has_text_ratio = df_clean["has_text"].mean()
print(f"\nProporción de juegos con alguna descripción de texto utilizable: {has_text_ratio:.2%}")

# Preparamos columnas a mostrar (solo las que existan)
cols_to_show = [c for c in ["detailed_description", "short_description", "text_description", "has_text"] if c in df_clean.columns]

print("\nEjemplos de juegos SIN texto (si existen):")
display(df_clean.loc[~df_clean["has_text"], cols_to_show].head())

print("\nEjemplos de juegos CON texto combinado:")
display(df_clean.loc[df_clean["has_text"], cols_to_show].head())

## 5. Conversión de `release_date` a formato fecha (`datetime`)

La columna `release_date` viene como **texto** con formato estilo inglés, por ejemplo:

- `"Aug 1, 2023"`
- `"Jul 29, 2016"`

En algunos casos también puede haber valores especiales como:

- `"Coming Soon"`
- `"TBA"`
- Fechas poco definidas.

### 5.1 Objetivo

Queremos convertir `release_date` a un tipo de dato de fecha (`datetime` en pandas) para poder:

- Ordenar juegos por fecha de lanzamiento.
- Filtrar por juegos “recientes” o “clásicos”.
- Hacer gráficos de número de lanzamientos por año, etc.

### 5.2 Estrategia de conversión

1. **Intentar parsear todas las fechas con un formato flexible**  
   - Usaremos funciones de pandas (`pd.to_datetime`) que permiten interpretar las fechas
     automáticamente en la mayoría de los casos.

2. **Tratar valores no válidos (TBA, Coming Soon, etc.)**  
   - Estos valores **no representan una fecha real**.
   - Durante la conversión los transformaremos en `NaT` (Not a Time), que es
     el equivalente de “valor nulo” para fechas en pandas.

3. **Mantener la información de que una fecha es desconocida**  
   - No eliminaremos automáticamente los juegos sin fecha válida.
   - Simplemente sabremos que `release_date` es desconocida y, si es necesario,
     podremos filtrarlos en análisis posteriores.

### 5.3 Resultado esperado

- Una columna `release_date` de tipo `datetime64` en el dataframe limpio.
- Los valores no interpretables quedarán como `NaT`.
- Esto nos permitirá hacer:

  - filtrados por rango de años,
  - análisis de tendencias temporales,
  - posibles filtros en la app (ej. “solo juegos a partir de 2015”).

In [ ]:
# 5. Conversión de 'release_date' a tipo datetime

# Comprobamos que la columna exista
if "release_date" in df_clean.columns:
    print("Antes de la conversión, tipo de 'release_date':", df_clean["release_date"].dtype)

    # Vemos algunos valores distintos para hacernos una idea del formato
    print("\nEjemplos de valores de 'release_date' (no nulos):")
    display(
        df_clean["release_date"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(10)
        .to_frame(name="release_date_raw_example")
    )

    # Guardamos una copia del texto original por si queremos inspeccionarlo después
    df_clean["release_date_raw"] = df_clean["release_date"].astype(str)

    # Convertimos a datetime:
    # - errors='coerce' -> valores no interpretables pasan a NaT
    # - dayfirst=False porque el formato original es estilo inglés (mes primero)
    df_clean["release_date"] = pd.to_datetime(
        df_clean["release_date_raw"],
        errors="coerce",
        dayfirst=False,
    )

    # Creamos una columna de año de lanzamiento (puede ser útil para EDA y filtros)
    df_clean["release_year"] = df_clean["release_date"].dt.year

    # Resumen rápido después de la conversión
    print("\nDespués de la conversión:")
    print("Tipo de 'release_date':", df_clean["release_date"].dtype)
    print("Nº de valores NaT (fechas no interpretables):", df_clean["release_date"].isna().sum())

    # Ejemplos de filas con fecha válida
    print("\nEjemplos de juegos con fecha de lanzamiento válida:")
    display(
        df_clean.loc[df_clean["release_date"].notna(), ["name", "release_date_raw", "release_date", "release_year"]]
        .head()
    )

    # Ejemplos de filas con fecha no interpretada (NaT)
    print("\nEjemplos de juegos con fecha no interpretada (NaT):")
    display(
        df_clean.loc[df_clean["release_date"].isna(), ["name", "release_date_raw", "release_date"]]
        .head()
    )

else:
    print("Aviso: no se encontró la columna 'release_date' en df_clean.")

## 6. Conversión de `tags` en una estructura usable (listas)

La columna `tags` contiene información muy rica sobre cada juego. Ejemplos de etiquetas son:

- `"Rogue-lite"`, `"Souls-like"`, `"Pixel Graphics"`, `"Story Rich"`, `"Co-op"`, etc.

En el dataset original, `tags` es un **listado de etiquetas** (lista de strings). Sin embargo,
según cómo se haya guardado y leído el archivo `games.parquet`, es posible que en nuestro
dataframe actual aparezca como:

- una lista Python (`list`)
- o un **texto** que representa un diccionario/JSON.

Por ejemplo:

- `["Rogue-lite", "Souls-like", "Pixel Graphics"]`
- o algo similar a `"{'Rogue-lite': 123, 'Pixel Graphics': 45}"`

### 6.1 Objetivo

Queremos transformar `tags` en una **lista limpia de etiquetas de texto** para cada juego, algo así:

```text
["rogue-lite", "souls-like", "pixel graphics"]

In [ ]:
# 6. Conversión de 'tags' a listas limpias y creación de 'tags_text'

import ast

# Inspeccionamos brevemente cómo vienen los datos en 'tags'
if "tags" in df_clean.columns:
    print("Muestras de la columna 'tags':")
    display(df_clean["tags"].head())

    print("\nTipos más frecuentes en la columna 'tags':")
    display(df_clean["tags"].apply(lambda x: type(x).__name__).value_counts().head())
else:
    print("Aviso: no se encontró la columna 'tags' en df_clean.")


def parse_tags(value):
    """
    Normaliza la columna 'tags' en una lista de strings en minúsculas.

    Casos contemplados:
    - value es NaN -> []
    - value es list/tuple/set/ndarray -> se convierte a lista
    - value es un string que representa un dict o una lista -> intentamos parsearlo con ast.literal_eval
    - value es un string plano (p.ej. etiquetas separadas por comas) -> lo partimos por comas

    Siempre devolvemos: lista de strings en minúsculas, sin espacios sobrantes y sin duplicados.
    """
    # 1) Nulos -> lista vacía
    if pd.isna(value):
        return []

    # 2) Si ya es una colección (lista, tupla, set, ndarray)
    if isinstance(value, (list, tuple, set, np.ndarray)):
        tags_list = list(value)

    else:
        # 3) Convertimos a string y limpiamos espacios
        s = str(value).strip()
        if not s:
            return []

        parsed = None

        # Solo intentamos literal_eval si el string parece una estructura (empieza por { o [)
        if s[0] in "{[":
            try:
                parsed = ast.literal_eval(s)
            except (ValueError, SyntaxError):
                parsed = None

        if isinstance(parsed, dict):
            # Nos quedamos con las claves del diccionario (nombres de etiquetas)
            tags_list = list(parsed.keys())
        elif isinstance(parsed, (list, tuple, set, np.ndarray)):
            tags_list = list(parsed)
        else:
            # Fallback: suponemos que son etiquetas separadas por comas
            tags_list = [t.strip() for t in s.split(",") if t.strip()]

    # 4) Normalización final: todo a string, minúsculas, sin vacíos, sin duplicados
    clean_tags = []
    seen = set()

    for t in tags_list:
        if not isinstance(t, str):
            t = str(t)
        t = t.strip().lower()
        if t and t not in seen:
            clean_tags.append(t)
            seen.add(t)

    return clean_tags


if "tags" in df_clean.columns:
    # Aplicamos la función de normalización
    df_clean["tags"] = df_clean["tags"].apply(parse_tags)

    # Creamos una versión en texto unificado para modelos de texto (TF-IDF, etc.)
    df_clean["tags_text"] = df_clean["tags"].apply(lambda tags: "; ".join(tags))

    print("\nEjemplos de conversión de 'tags':")
    display(df_clean[["name", "tags", "tags_text"]].head())
else:
    print("Aviso: no se encontró la columna 'tags'; no se aplica la conversión.")

## 7. Columnas que descartamos en el dataset limpio

Pensando en el **MVP del sistema de recomendación**, hay columnas que no aportan
valor directo o que añaden mucha complejidad/tamaño sin un beneficio claro.

### 7.1 Columnas claramente descartadas

Estas columnas no se incluirán en `games_clean.parquet`:

- `support_url`  
  URL de soporte técnico. No aporta información sobre el contenido del juego ni los gustos del usuario.

- `support_email`  
  Email de soporte. Ídem anterior.

- `metacritic_url`  
  Enlace a la web de Metacritic. La información importante es la puntuación, no la URL.

- `notes`  
  Campo textual poco estructurado y muy variable. Introduce ruido y es difícil de usar de forma sistemática.

- `movies`  
  Información de trailers/vídeos. Añade complejidad y tamaño, y no es esencial para este proyecto.

- `packages`  
  Información de paquetes/bundles comerciales. Más cercano a la lógica comercial de Steam que a los gustos del usuario.

- Columnas técnicas como `__index_level_0__` u otras generadas al guardar/leer Parquet  
  No representan información del juego. Son artefactos internos y se eliminan siempre.

### 7.2 Columnas “opcionales” que simplificamos

Algunas columnas contienen información útil pero parcialmente redundante.
Para mantener el proyecto manejable, tomaremos decisiones razonables, por ejemplo:

- Entre `average_playtime_forever` y `median_playtime_forever`  
  > Nos quedamos solo con **una** de las dos para representar el “engagement” del juego
  (por ejemplo, `average_playtime_forever`).

- Entre `average_playtime_2weeks` y `median_playtime_2weeks`  
  > Son métricas de muy corto plazo. Si no vamos a hacer análisis de “actividad reciente”,
  podemos eliminarlas para simplificar el dataset.

- `score_rank`  
  > Resume en texto el orden relativo del juego. Es redundante con `user_score` y las reseñas
  (`positive`/`negative`/`recommendations`), por lo que podemos eliminarla en la versión limpia.

### 7.3 Resumen

En el dataset limpio `games_clean.parquet` mantendremos:

- Identificadores y metadatos básicos:  
  `appID`, `name`, `release_date`, `release_year`, `price`, `required_age`, `windows`, `mac`, `linux`.

- Texto descriptivo:  
  `detailed_description`, `short_description`, `reviews` (y la columna combinada `text_description`).

- Métricas de popularidad/calidad:  
  `estimated_owners`, `peak_ccu`, `user_score`, `positive`, `negative`, `recommendations`,
  (y una métrica de tiempo de juego, p. ej. `average_playtime_forever`).

- Metadatos de contenido:  
  `genres`, `tags`, `tags_text`, `categories`, `developers`, `publishers`,
  `supported_languages`, `full_audio_languages`.

- Elementos visuales mínimos:  
  `header_image`, `website` (opcional).

Todo lo demás se elimina o se marca como “no prioritario”, para que el resto del equipo pueda
trabajar con un dataset:

- más ligero,
- más coherente,
- y centrado en lo que realmente afecta a las recomendaciones.

In [ ]:
# 7A. Definimos las columnas que queremos eliminar y aplicamos el drop

from pathlib import Path

# Lista de columnas a eliminar
columns_to_drop = [
    # Claramente descartadas
    "support_url",
    "support_email",
    "metacritic_url",
    "notes",
    "movies",
    "packages",
    "__index_level_0__",   # columna técnica de índice, si existiera

    # Opcionales que simplificamos
    "score_rank",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
]

# Nos quedamos solo con las que realmente existen en df_clean
cols_present = [c for c in columns_to_drop if c in df_clean.columns]

print("Columnas que se van a eliminar en el dataset limpio:")
print(cols_present)

# Aplicamos el drop
df_clean = df_clean.drop(columns=cols_present)

print("\nDimensiones de df_clean después de eliminar columnas:", df_clean.shape)

In [ ]:
# 7B. Aseguramos tipos numéricos en columnas clave (operación ligera)

numeric_cols = [
    "price",
    "metacritic_score",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "average_playtime_forever",
    "required_age",
    "dlc_count",
    # "estimated_owners",  # la quitamos de aquí para evitar problemas si es rango tipo "0 - 20000"
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("\nConversión numérica completada para columnas presentes.")
print("Ejemplo de tipos de datos (primeras 30 columnas):")
display(df_clean.dtypes.to_frame("dtype").sort_index().head(30))

In [ ]:
file_path = f"{PROJECT_ROOT}/data/processed/df_clean.parquet"

with open(file_path, "wb") as f:
    pickle.dump(df_clean, f)
    print(f"Cleaned DataFrame saved in: {file_path}")

## 8. Pipeline de limpieza como módulo reutilizable

Toda la lógica de limpieza que hemos explorado en este cuaderno se ha encapsulado en:

- `src/data/load_data.py`: carga y descarga los datos brutos y limpios.
- `src/data/clean_data.py`: aplica el pipeline de limpieza al dataset de Steam.

De esta forma, el resto del proyecto (EDA, modelos, app) puede reutilizar
el mismo pipeline sin copiar código desde el notebook.

In [ ]:
from src.data.load_data import load_raw_data, load_clean_data
from src.data.clean_data import build_and_save_clean_dataset

# 1) Cargar datos brutos
df_raw = load_raw_data()
print("df_raw:", df_raw.shape)

# 2) Construir y guardar dataset limpio
df_clean_built = build_and_save_clean_dataset()
print("df_clean_built:", df_clean_built.shape)

# 3) Cargar dataset limpio directamente desde processed
df_clean = load_clean_data()
print("df_clean (desde archivo limpio):", df_clean.shape)

df_clean.head()